# SEC API Data Exploration Starter

This notebook is focused on U.S. Securities and Exchange Commission (SEC) data extraction only. Scoring has been intentionally deferred for a later workflow.

How to use this notebook:

- Each step has a markdown section that explains the purpose, expected inputs, and expected outputs.
- Each code section produces an extraction table, raw-data preview, or non-scoring financial summary used by later analysis.
- Run the notebook top to bottom during exploration, then promote stable pieces into pipeline code when ready.

Primary extraction outputs:

- `sec_company_crosswalk`: starter bridge between SEC entities and USAspending identifiers.
- `sec_company_submissions`: filing/entity metadata from SEC submissions endpoints.
- `sec_company_facts_long`: selected raw XBRL fact-level time series.
- `sec_company_financials_wide`: non-scoring company-year financial summary.


## Acronym Glossary

- Securities and Exchange Commission (SEC)
- Application Programming Interface (API)
- Central Index Key (CIK)
- Unique Entity Identifier (UEI)
- Data Universal Numbering System (DUNS)
- eXtensible Business Reporting Language (XBRL)
- Standard Industrial Classification (SIC)
- Form 10-K (annual report), Form 10-Q (quarterly report), Form 20-F and 40-F (non-U.S. annual filings)
- Comma-Separated Values (CSV)
- Business Intelligence (BI)
- Quality Assurance (QA)


## Step 1: Environment Setup

Purpose:
Create a single runtime configuration that all later cells use (imports, request pacing, output directory, and identity headers).

Why this matters:
- Consistent setup prevents hidden differences between cells and makes reruns reproducible.
- The `SEC_USER_AGENT` value identifies your script to the SEC. The SEC expects automated clients to identify themselves under fair-access guidance. Using a clear organization/contact string reduces the risk of request throttling or blocking and makes your traffic attributable.
- Creating `OUTPUT_DIR` at startup prevents later file-write failures when exporting tables.

What this section initializes:
- `SEC_USER_AGENT`: request identity string.
- `REQUEST_DELAY_SECONDS`: minimum pause between API calls.
- `OUTPUT_DIR`: target directory for generated CSV artifacts.


In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path

import pandas as pd
import requests

SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "KCNSC (aarango@kcnsc.doe.com)")
REQUEST_DELAY_SECONDS = 0.2
OUTPUT_DIR = Path("output") / "sec_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not SEC_USER_AGENT.strip():
    raise ValueError("SEC_USER_AGENT must not be empty.")


## Step 2: SEC Request Helper

Purpose:
Define one reusable request path for SEC JSON endpoints.

Why this matters:
- Centralizing request logic ensures every call uses the same headers, timeout, and delay behavior.
- A single helper function (`sec_get_json`) makes it easier to add retries/logging later without changing many cells.
- Throttling in one place is safer than relying on manual delays spread across cells.

What this section creates:
- `build_sec_session(...)`: prepares a `requests.Session` with common headers.
- `sec_get_json(...)`: applies delay, constructs URL, executes request, raises on HTTP errors, and returns parsed JSON.


In [ ]:
BASE_SEC_URL = "https://data.sec.gov"
_LAST_SEC_CALL_TS = 0.0

def build_sec_session(user_agent: str) -> requests.Session:
    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": user_agent,
            "Accept-Encoding": "gzip, deflate",
        }
    )
    return session

sec_session = build_sec_session(SEC_USER_AGENT)

def sec_get_json(path_or_url: str, pause_seconds: float = REQUEST_DELAY_SECONDS) -> dict:
    global _LAST_SEC_CALL_TS
    now = time.time()
    elapsed = now - _LAST_SEC_CALL_TS
    if elapsed < pause_seconds:
        time.sleep(pause_seconds - elapsed)

    url = path_or_url if path_or_url.startswith("http") else f"{BASE_SEC_URL}{path_or_url}"
    response = sec_session.get(url, timeout=45)
    _LAST_SEC_CALL_TS = time.time()
    response.raise_for_status()
    return response.json()


## Step 3: Ticker-to-CIK Mapping

Purpose:
Build the canonical lookup from ticker symbol to Central Index Key (CIK).

Why this matters:
- Most SEC company endpoints require CIK, not ticker.
- Analysts typically start with tickers, so this mapping is the bridge between analyst inputs and SEC endpoint requirements.
- Zero-padding CIK values to 10 digits avoids join mismatches when merging datasets.

What to validate after running:
- `ticker_map` is not empty.
- `ticker` values are uppercase.
- `cik` values are 10-character strings.


In [ ]:
# Pull SEC ticker-to-CIK mapping
# Use JSON endpoints only.
ticker_sources = [
    "https://www.sec.gov/files/company_tickers.json",
    "https://sec.gov/files/company_tickers.json",
]

ticker_payload = None
last_exc = None
for source in ticker_sources:
    try:
        ticker_payload = sec_get_json(source)
        break
    except Exception as exc:
        last_exc = exc

if ticker_payload is None:
    raise RuntimeError(f"Unable to load SEC ticker mapping from JSON endpoints: {last_exc}")

ticker_map = pd.DataFrame.from_dict(ticker_payload, orient="index")
ticker_map = ticker_map.rename(columns={"cik_str": "cik", "title": "company_name"})
ticker_map["cik"] = ticker_map["cik"].astype(int).astype(str).str.zfill(10)
ticker_map["ticker"] = ticker_map["ticker"].str.upper()
ticker_map = ticker_map[["ticker", "cik", "company_name"]].sort_values("ticker").reset_index(drop=True)
ticker_map.head()


## Step 4: Crosswalk Seed Table

Purpose:
Create an initial company bridge table that can eventually connect SEC data to USAspending entities.

Why this matters:
- There is no universal direct key between SEC entities and USAspending entities.
- A crosswalk table provides an explicit, auditable place to store matching decisions over time.
- Tracking `match_method`, `match_confidence`, and `last_verified_date` supports governance and quality review.

What this section produces:
- `sec_company_crosswalk` with CIK/ticker plus placeholder UEI/DUNS fields for future enrichment.


In [ ]:
# Seed set can be replaced with your current target companies
SEED_TICKERS = ["LMT", "RTX", "BA"]

sec_company_crosswalk = (
    ticker_map[ticker_map["ticker"].isin(SEED_TICKERS)]
    .drop_duplicates(subset=["ticker"])
    .assign(
        uei=pd.NA,
        duns=pd.NA,
        recipient_name=lambda df: df["company_name"],
        match_method="manual_seed",
        match_confidence=1.0,
        last_verified_date=pd.Timestamp.today().normalize(),
    )
    [[
        "uei",
        "duns",
        "recipient_name",
        "cik",
        "ticker",
        "match_method",
        "match_confidence",
        "last_verified_date",
    ]]
    .sort_values("ticker")
    .reset_index(drop=True)
)

sec_company_crosswalk


## Step 5: Optional USAspending Context

Purpose:
Optionally load `entity_master.csv` from your USAspending workflow.

Why this matters:
- If present, it provides the target universe for matching and validation.
- If absent, the notebook still works in SEC-only mode so development is not blocked.
- This pattern lets new developers test logic without requiring full upstream data every time.

Expected behavior:
- Prints row count when file exists.
- Prints SEC-only continuation message when file is missing.


In [ ]:
# Optional: load entity_master if you already created it from USAspending
entity_master_path = Path("entity_master.csv")
if entity_master_path.exists():
    entity_master = pd.read_csv(entity_master_path, dtype=str)
    print(f"entity_master rows: {len(entity_master):,}")
else:
    entity_master = pd.DataFrame()
    print("entity_master.csv not found yet. Continue with SEC-only exploration.")


## Step 6: Pull SEC Submissions and Facts

Purpose:
Retrieve both metadata and numeric fact data for each seeded company.

Why this matters:
- `submissions` data contains company-level metadata (name, industry code, fiscal year end, etc.).
- This includes Standard Industrial Classification (SIC) fields: `sic` is the numeric industry code, and `sic_description` is the corresponding plain-language industry label.
- `companyfacts` data contains selected raw XBRL fact-level time series for later analysis.
- Keeping a long-form raw facts table preserves detail for QA before transformation.

What this section does:
- Defines financial tags to collect (revenue variants, net income, assets, liabilities).
- Calls submissions and companyfacts endpoints per company.
- Builds:
  - `sec_company_submissions` (metadata table)
  - `sec_company_facts_long` (raw fact table)

Expected quick check:
- `submissions rows` should generally equal seeded company count (minus failed pulls).
- `facts rows` should be substantially larger than submissions rows.


In [ ]:
FACT_TAGS = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "SalesRevenueNet",
    "NetIncomeLoss",
    "Assets",
    "Liabilities",
]

def extract_usd_facts(facts_json: dict, cik: str, ticker: str) -> list[dict]:
    rows: list[dict] = []
    us_gaap = facts_json.get("facts", {}).get("us-gaap", {})

    for tag in FACT_TAGS:
        units = us_gaap.get(tag, {}).get("units", {})
        usd_points = units.get("USD", [])

        for point in usd_points:
            rows.append(
                {
                    "cik": cik,
                    "ticker": ticker,
                    "tag": tag,
                    "end": point.get("end"),
                    "start": point.get("start"),
                    "val": point.get("val"),
                    "fy": point.get("fy"),
                    "fp": point.get("fp"),
                    "form": point.get("form"),
                    "filed": point.get("filed"),
                    "accn": point.get("accn"),
                }
            )

    return rows

submission_rows = []
facts_rows = []

for row in sec_company_crosswalk.itertuples(index=False):
    cik = str(row.cik).zfill(10)
    ticker = row.ticker

    try:
        submission = sec_get_json(f"/submissions/CIK{cik}.json")
        facts = sec_get_json(f"/api/xbrl/companyfacts/CIK{cik}.json")
    except requests.HTTPError as exc:
        print(f"Skipping {ticker} ({cik}) due to HTTP error: {exc}")
        continue

    submission_rows.append(
        {
            "cik": cik,
            "ticker": ticker,
            "entity_name": submission.get("name"),
            "sic": submission.get("sic"),
            "sic_description": submission.get("sicDescription"),
            "fiscal_year_end": submission.get("fiscalYearEnd"),
            "state_of_incorporation": submission.get("stateOfIncorporation"),
            "phone": submission.get("phone"),
            "entity_type": submission.get("entityType"),
        }
    )

    facts_rows.extend(extract_usd_facts(facts, cik=cik, ticker=ticker))

sec_company_submissions = pd.DataFrame(submission_rows)
sec_company_facts_long = pd.DataFrame(facts_rows).rename(columns={"val": "value"})

print(f"submissions rows: {len(sec_company_submissions):,}")
print(f"facts rows: {len(sec_company_facts_long):,}")


## Step 6B: Company Profile Metadata Dictionary

Purpose:
Document the company profile fields collected from the SEC submissions endpoint so users can interpret exported metadata correctly.

Metadata fields collected in `sec_company_submissions`:
- `cik`: Central Index Key (CIK), the SEC company identifier.
- `ticker`: Public market ticker symbol used in the seed list.
- `entity_name`: Legal entity name returned by SEC submissions.
- `sic`: Standard Industrial Classification (SIC) numeric industry code.
- `sic_description`: Plain-language industry name associated with the SIC code.
- `fiscal_year_end`: Fiscal year-end in `MMDD` format (month/day, no separator).
  - Example: `1231` means the company fiscal year ends on December 31.
  - Example: `0630` means the company fiscal year ends on June 30.
- `state_of_incorporation`: State code where the entity is incorporated.
- `phone`: Company contact number as reported in submissions metadata.
- `entity_type`: SEC entity type classification (for example `operating`).

Why this matters:
- These fields provide business context around the facts table and are useful for QA, filtering, and reporting.
- `fiscal_year_end` is especially important for interpreting annual comparisons across companies with different reporting calendars.


## Step 7: Raw Facts Sanity Check

Purpose:
Preview raw fact rows to confirm the SEC extraction succeeded.

Why this matters:
- Early validation catches schema or parsing issues before they propagate.
- You can confirm key fields are present (`cik`, `ticker`, `tag`, `value`, `fy`, `form`, `filed`).
- This step confirms that the raw feed contains enough historical depth for trend calculations.

Important note:
- `entity_name` does not live in `sec_company_facts_long`; it is part of `sec_company_submissions` metadata.


In [ ]:
sec_company_facts_long.head()


## Step 7B: Entity Name Verification Join

Purpose:
Confirm where `entity_name` is stored and demonstrate how to attach it to raw fact rows.

Why this matters:
- New developers often expect `entity_name` in the raw facts table; in this pipeline, names come from submissions metadata.
- This join pattern is the standard way to enrich fact-level tables with company display fields for QA previews and reporting.

What this section does:
- Shows `cik`, `ticker`, and `entity_name` directly from `sec_company_submissions`.
- Left-joins names into `sec_company_facts_long` to produce a preview table with both facts and company names.


In [ ]:
# Verify where entity_name lives and optionally join it to facts preview
sec_company_submissions[["cik", "ticker", "entity_name"]]

facts_with_names = sec_company_facts_long.merge(
    sec_company_submissions[["cik", "entity_name"]],
    on="cik",
    how="left"
)
facts_with_names.head()


## Step 8: Build Non-Scoring Financial Summary

Purpose:
Create a readable company-year financial summary from the raw SEC facts without assigning scores or ratings.

Why this matters:
- `sec_company_facts_long` is the source-level extraction table, where each row is one SEC fact.
- Analysts often need a wider table where revenue, net income, assets, and liabilities are visible as separate columns.
- This table keeps the data transformation descriptive only; scoring can be added later in a separate workflow.

What this section produces:
- `sec_company_financials_wide`: one row per CIK/ticker/fiscal year with selected financial values as columns.


In [ ]:
ANNUAL_FORMS = {"10-K", "20-F", "40-F"}

financial_facts = sec_company_facts_long[sec_company_facts_long["form"].isin(ANNUAL_FORMS)].copy()
financial_facts["fiscal_year"] = pd.to_numeric(financial_facts["fy"], errors="coerce")
financial_facts["filed_date"] = pd.to_datetime(financial_facts["filed"], errors="coerce")
financial_facts = financial_facts.dropna(subset=["fiscal_year"])
financial_facts["fiscal_year"] = financial_facts["fiscal_year"].astype(int)

# Keep the latest reported value for each company, tag, and fiscal year.
financial_facts = (
    financial_facts
    .sort_values(["cik", "tag", "fiscal_year", "filed_date"])
    .drop_duplicates(subset=["cik", "ticker", "tag", "fiscal_year"], keep="last")
)

sec_company_financials_wide = (
    financial_facts
    .pivot_table(index=["cik", "ticker", "fiscal_year"], columns="tag", values="value", aggfunc="last")
    .reset_index()
)
sec_company_financials_wide.columns.name = None

for column in FACT_TAGS:
    if column not in sec_company_financials_wide.columns:
        sec_company_financials_wide[column] = pd.NA

revenue_source_columns = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "SalesRevenueNet",
]
available_revenue_columns = [column for column in revenue_source_columns if column in sec_company_financials_wide.columns]
if available_revenue_columns:
    sec_company_financials_wide["reported_revenue"] = sec_company_financials_wide[available_revenue_columns].bfill(axis=1).iloc[:, 0]
else:
    sec_company_financials_wide["reported_revenue"] = pd.NA

sec_company_financials_wide = sec_company_financials_wide.rename(
    columns={
        "Revenues": "revenues",
        "RevenueFromContractWithCustomerExcludingAssessedTax": "revenue_from_contract",
        "SalesRevenueNet": "sales_revenue_net",
        "NetIncomeLoss": "net_income_loss",
        "Assets": "assets",
        "Liabilities": "liabilities",
    }
)

company_profile_columns = ["cik", "entity_name", "sic_description", "fiscal_year_end"]
sec_company_financials_wide = sec_company_financials_wide.merge(
    sec_company_submissions[company_profile_columns].drop_duplicates(subset=["cik"]),
    on="cik",
    how="left",
)

preferred_columns = [
    "cik",
    "ticker",
    "entity_name",
    "sic_description",
    "fiscal_year_end",
    "fiscal_year",
    "reported_revenue",
    "revenues",
    "revenue_from_contract",
    "sales_revenue_net",
    "net_income_loss",
    "assets",
    "liabilities",
]
sec_company_financials_wide = sec_company_financials_wide[preferred_columns].sort_values(["ticker", "fiscal_year"]).reset_index(drop=True)
sec_company_financials_wide.head()




## Step 9: Export Extracted Tables

Export the extracted SEC working tables and the non-scoring financial summary. Scoring outputs are intentionally excluded for now.

Files are written to `output/sec_exploration/`.

Exported files:

1. `sec_company_crosswalk.csv`
- What it contains: Starter bridge records linking SEC identifiers (`cik`, `ticker`) with placeholders for USAspending identifiers (`uei`, `duns`), plus match governance fields.
- Intended use: Maintain an auditable crosswalk process between SEC company records and USAspending entities.

2. `sec_company_submissions.csv`
- What it contains: Company-level metadata from SEC submissions endpoints, such as `entity_name`, Standard Industrial Classification (SIC) fields (`sic` and `sic_description`), `fiscal_year_end`, and related profile fields.
- Intended use: Company profile staging table for later matching, review, and enrichment.

3. `sec_company_facts_long.csv`
- What it contains: Selected raw XBRL fact rows by CIK, ticker, SEC tag, period, form, filing date, accession number, and value.
- Intended use: Source-of-truth staging layer before future analysis work.

4. `sec_company_financials_wide.csv`
- What it contains: Non-scoring company-year financial values with revenue, net income, assets, and liabilities as columns.
- Intended use: Analyst-readable financial summary before any future scoring model is applied.


In [ ]:
table_exports = {
    "sec_company_crosswalk.csv": sec_company_crosswalk,
    "sec_company_submissions.csv": sec_company_submissions,
    "sec_company_facts_long.csv": sec_company_facts_long,
    "sec_company_financials_wide.csv": sec_company_financials_wide,
}

for filename, frame in table_exports.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

print(f"Wrote {len(table_exports)} extraction tables to: {OUTPUT_DIR.resolve()}")
list(table_exports.keys())
